DATA CLEANING 

In [17]:
import numpy as np
import pandas as pd
import re


In [3]:
df=pd.read_csv('../data/raw/Books_Raw_Data.csv')
print(f"Shape:{df.shape}")
df.head(2)

Shape:(20068, 24)


,Unnamed: 0,bookId,title,author,series,description,genres,awards,characters,places,...,publish_date,num_pages,num_ratings,num_reviews,avg_rating,rated_1,rated_2,rated_3,rated_4,rated_5
0,0,1,Harry Potter and the Half-Blood Prince,J.K. Rowling,Harry Potter #6,The war against Voldemort is not going well; e...,"Art,Biography,Business,Children's,Christian,Cl...",Locus Award Nominee for Best Young Adult Novel...,"Draco Malfoy,Ron Weasley,Petunia Dursley,Verno...","Hogwarts School of Witchcraft and Wizardry,Eng...",...,September 16th 2006,652.0,2553909.0,41470.0,4.57,13147.0,29020.0,174312.0,608825.0,1728605.0
1,1,2,Harry Potter and the Order of the Phoenix,"J.K. Rowling,Mary GrandPré",Harry Potter #5,There is a door at the end of a silent corrido...,"Art,Biography,Business,Children's,Christian,Cl...",Bram Stoker Award for Works for Young Readers ...,"Sirius Black,Draco Malfoy,Ron Weasley,Petunia ...","Hogwarts School of Witchcraft and Wizardry,Lon...",...,September 2004,870.0,2631427.0,44793.0,4.50,16236.0,41738.0,231438.0,665628.0,1676387.0


In [4]:
print(df.columns)

Index(['Unnamed: 0', 'bookId', 'title', 'author', 'series', 'description',
       'genres', 'awards', 'characters', 'places', 'isbn', 'isbn13',
       'language', 'first_publish_date', 'publish_date', 'num_pages',
       'num_ratings', 'num_reviews', 'avg_rating', 'rated_1', 'rated_2',
       'rated_3', 'rated_4', 'rated_5'],
      dtype='str')


In [5]:
print(f"{'Column':<22} {'Dtype':<10} {'Non-Null':>9}  {'% Missing':>10}")
print("─" * 58)
for col in df.columns:
    n_miss = df[col].isna().sum()
    pct    = 100 * n_miss / len(df)
    print(f"{col:<22} {str(df[col].dtype):<10} {len(df)-n_miss:>9,}  {pct:>9.1f}%")

Column                 Dtype       Non-Null   % Missing
──────────────────────────────────────────────────────────
Unnamed: 0             int64         20,068        0.0%
bookId                 int64         20,068        0.0%
title                  str           19,509        2.8%
author                 str           19,507        2.8%
series                 str            4,279       78.7%
description            str           17,381       13.4%
genres                 str           19,812        1.3%
awards                 str            3,129       84.4%
characters             str            4,779       76.2%
places                 str            4,040       79.9%
isbn                   str           17,531       12.6%
isbn13                 str           19,210        4.3%
language               str           17,289       13.8%
first_publish_date     str           14,591       27.3%
publish_date           str           19,022        5.2%
num_pages              float64       18,428  

we can observe that there is an significant amount of missing % in the columns

In [6]:
# Numeric summary
num_cols = df.select_dtypes(include='number').columns.tolist()
df[num_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,20068.0,10033.50,5793.28,0.0,5016.75,10033.50,15050.25,20067.0
bookId,20068.0,9980.30,5762.99,1.0,5014.75,9937.50,14968.25,20000.0
num_pages,18428.0,292.45,307.33,0.0,148.00,252.00,369.00,23931.0
num_ratings,19509.0,74103.52,330281.16,0.0,20.00,437.00,11146.00,7444102.0
num_reviews,19509.0,2073.16,7394.39,0.0,2.00,32.00,567.00,117992.0
avg_rating,19509.0,3.69,1.00,0.0,3.67,3.93,4.14,5.0
rated_1,16638.0,2269.43,10663.29,0.0,1.00,17.00,323.75,562807.0
rated_2,16638.0,4625.52,18495.32,0.0,4.00,56.00,979.00,560978.0
rated_3,16638.0,15705.20,57986.17,0.0,17.00,242.00,3945.50,1052619.0
rated_4,16638.0,27801.18,106233.32,0.0,26.00,380.00,6686.75,1696235.0


In [7]:
# 'Unnamed: 0' is a leftover CSV row-index — not useful
df = df.drop(columns=['Unnamed: 0'])
print("Dropped 'Unnamed: 0' ✓  |  Shape:", df.shape)

Dropped 'Unnamed: 0' ✓  |  Shape: (20068, 23)


In [9]:
#drop rows with no title , author , rating
before = len(df)
df = df.dropna(subset=['title', 'author', 'avg_rating'])
print(f"Dropped {before - len(df):,} rows missing title / author / avg_rating")
print(f"Remaining : {len(df):,}")

Dropped 0 rows missing title / author / avg_rating
Remaining : 19,507


In [10]:
#remove books with 0ratings or 0 ayg rating
# avg_rating == 0  →  not yet rated, useless for similarity ranking
# num_ratings == 0  →  same issue
before = len(df)
df = df[(df['avg_rating'] > 0) & (df['num_ratings'] > 0)]
print(f"Removed {before - len(df):,} unrated books  |  Remaining : {len(df):,}")

Removed 1,125 unrated books  |  Remaining : 18,382


In [11]:
#handling duplicates
print("bookId duplicates      :", df['bookId'].duplicated().sum())
print("Title + Author dups   :", df.duplicated(subset=['title','author']).sum())
print("Full-row duplicates   :", df.duplicated().sum())

bookId duplicates      : 122
Title + Author dups   : 1376
Full-row duplicates   : 119


In [12]:
# For bookId duplicates — keep the one with more ratings (more popular edition)
df = df.sort_values('num_ratings', ascending=False)
df = df.drop_duplicates(subset='bookId', keep='first')

# For same title+author (different editions) — also keep highest-rated edition
df = df.drop_duplicates(subset=['title', 'author'], keep='first')

df = df.reset_index(drop=True)
print(f"After dedup : {len(df):,} rows")

After dedup : 17,006 rows


In [13]:
#language keeping english only
print("Language distribution (top 10):")
print(df['language'].value_counts().head(10))

Language distribution (top 10):
language
English                   14461
Spanish                     405
German                      173
French                      163
Japanese                     47
Multiple languages           22
Italian                      14
Chinese                      13
Portuguese                   13
Greek, Ancient to 1453       10
Name: count, dtype: int64


In [14]:
# Keep English and NaN (often English but unlabelled)
english_vals = ['English', 'English, Middle 1100-1500']  # Middle English included — it's rare but valid
before = len(df)
df = df[df['language'].isin(english_vals) | df['language'].isna()]
print(f"Dropped {before - len(df):,} non-English rows  |  Remaining : {len(df):,}")

Dropped 884 non-English rows  |  Remaining : 16,122


In [15]:
# fix genres clumn 
#Every row's genres string starts with a repeated list of ALL Goodreads shelf names.
# The actual book-specific genres come AFTER this prefix.
# Example: "Art,Biography,...,Young Adult,Art,Biography,...,Young Adult,Fantasy,Fiction,..."
#                       ^^^ two copies of the master list ^^^   ^^^ real genres start here ^^^

MASTER_PREFIX = (
    "Art,Biography,Business,Children's,Christian,Classics,Comics,Cookbooks,"
    "Ebooks,Fantasy,Fiction,Graphic Novels,Historical Fiction,History,Horror,"
    "Memoir,Music,Mystery,Nonfiction,Poetry,Psychology,Romance,Science,"
    "Science Fiction,Self Help,Sports,Thriller,Travel,Young Adult,"
)
DOUBLE_PREFIX = MASTER_PREFIX + MASTER_PREFIX   # sometimes appears twice

def extract_genres(raw):
    if pd.isna(raw):
        return ''
    g = str(raw)
    if g.startswith(DOUBLE_PREFIX):
        g = g[len(DOUBLE_PREFIX):]
    elif g.startswith(MASTER_PREFIX):
        g = g[len(MASTER_PREFIX):]
    # Remove duplicate genre tokens
    seen, unique = set(), []
    for token in g.split(','):
        t = token.strip()
        t_lower = t.lower()
        if t and t_lower not in seen and "goodreads" not in t_lower:
            seen.add(t_lower)
            unique.append(t)
    return ','.join(unique)

df['genres'] = df['genres'].apply(extract_genres)

# Verify
print("Sample cleaned genres:")
for i in range(5):
    print(f"  [{i}] {df['genres'].iloc[i][:100]}")

Sample cleaned genres:
  [0] Fantasy,Fiction,Young Adult,Magic,Childrens,Middle Grade,Adventure,Classics,Audiobook,Science Fictio
  [1] Fantasy,Fiction,Young Adult,Magic,Childrens,Middle Grade,Adventure,Classics,Audiobook,Science Fictio
  [2] Young Adult,Fantasy,Romance,Fiction,Paranormal,Vampires,Paranormal Romance,Supernatural,Teen,Urban F
  [3] Classics,Fiction,Historical,Historical Fiction,Academic,School,Literature,Young Adult,Novels,Read Fo
  [4] Classics,Fiction,Historical,Historical Fiction,Academic,School,Literature,Young Adult,Novels,Read Fo


In [18]:
#parse date columns to extract year
def extract_year(date_str):
    """Pull the 4-digit year from strings like 'July 16th 2005' or 'September 2004'."""
    if pd.isna(date_str):
        return np.nan
    match = re.search(r'\b(1[89]\d{2}|20[012]\d)\b', str(date_str))
    return int(match.group()) if match else np.nan

df['first_publish_year'] = df['first_publish_date'].apply(extract_year)
df['publish_year']       = df['publish_date'].apply(extract_year)

# Drop original messy date strings (we have the years now)
df = df.drop(columns=['first_publish_date', 'publish_date'])

print("Year extraction done.")
print(f"  first_publish_year nulls : {df['first_publish_year'].isna().sum():,}")
print(f"  publish_year nulls       : {df['publish_year'].isna().sum():,}")
print(f"  Year range (first pub)   : {int(df['first_publish_year'].min())} – {int(df['first_publish_year'].max())}")

Year extraction done.
  first_publish_year nulls : 4,567
  publish_year nulls       : 372
  Year range (first pub)   : 1800 – 2015


In [19]:
print(f"num_pages = 0  : {(df['num_pages'] == 0).sum():,} rows")
print(f"num_pages > 5000 : {(df['num_pages'] > 5000).sum():,} rows")
print(f"num_pages null   : {df['num_pages'].isna().sum():,} rows")

# Replace 0-page and extreme outlier rows with NaN, then fill with median
df.loc[df['num_pages'] == 0, 'num_pages'] = np.nan
df.loc[df['num_pages'] > 5000, 'num_pages'] = np.nan

median_pages = df['num_pages'].median()
df['num_pages'] = df['num_pages'].fillna(median_pages).astype(int)
print(f"\nFilled nulls with median = {int(median_pages)} pages")

num_pages = 0  : 519 rows
num_pages > 5000 : 1 rows
num_pages null   : 819 rows

Filled nulls with median = 256 pages


In [20]:
# description — empty string (will be handled during NLP preprocessing)
df['description'] = df['description'].fillna('')

# series — 'Standalone' for non-series books
df['series'] = df['series'].fillna('Standalone')

# awards / characters / places — empty string (optional enrichment fields)
for col in ['awards', 'characters', 'places']:
    df[col] = df[col].fillna('')

# language — 'English' (we kept only English + NaN rows above)
df['language'] = df['language'].fillna('English')

# isbn / isbn13 — keep as-is (not used in recommender core)
# publish_year — fill with first_publish_year where available
df['publish_year'] = df['publish_year'].fillna(df['first_publish_year'])
year_median = df['publish_year'].median()
df['publish_year'] = df['publish_year'].fillna(year_median).astype(int)

# rated_1..5 — fill with 0 (no votes for that star recorded)
for col in ['rated_1','rated_2','rated_3','rated_4','rated_5']:
    df[col] = df[col].fillna(0).astype(int)

# num_reviews
df['num_reviews'] = df['num_reviews'].fillna(0).astype(int)
df['num_ratings'] = df['num_ratings'].astype(int)

print("Null fill complete.")
print(df.isna().sum()[df.isna().sum() > 0])

Null fill complete.
isbn                  1608
isbn13                 202
first_publish_year    4567
dtype: int64


In [21]:
def clean_display_text(text):
    """Light clean for display columns: strip whitespace, collapse spaces."""
    if pd.isna(text): return ''
    return re.sub(r'\s+', ' ', str(text)).strip()

def clean_description(text):
    """Remove HTML tags, fix encoding artifacts, normalise whitespace."""
    if not text: return ''
    text = re.sub(r'<[^>]+>', ' ', text)          # strip HTML
    text = re.sub(r'&[a-z]+;', ' ', text)         # HTML entities
    text = re.sub(r'[^\w\s.,!?\'\'\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['title']       = df['title'].apply(clean_display_text)
df['author']      = df['author'].apply(clean_display_text)
df['description'] = df['description'].apply(clean_description)

print("Sample cleaned descriptions:")
for i in range(3):
    print(f"  [{i}] {df['description'].iloc[i][:120]}")

Sample cleaned descriptions:
  [0] When a letter arrives for unhappy but ordinary Harry Potter, a decade-old secret is revealed to him. His parents were wi
  [1] Harry Potter's life is miserable. His parents are dead and he's stuck with his heartless relatives, who force him to liv
  [2] About three things I was absolutely positive.,First, Edward was a vampire.,Second, there was a part of him and I didn't 


In [22]:
# Rating must be in [0.5, 5.0] for a published book
out_of_range = df[~df['avg_rating'].between(0.5, 5.0)]
print(f"avg_rating out of [0.5, 5.0]: {len(out_of_range)} rows")
df = df[df['avg_rating'].between(0.5, 5.0)]
print(f"Shape after rating filter: {df.shape}")

avg_rating out of [0.5, 5.0]: 0 rows
Shape after rating filter: (16122, 23)


In [23]:
# Bayesian average smooths low-count books toward the global mean
# score = (v/(v+m)) * R  +  (m/(v+m)) * C
# v = num_ratings, m = minimum threshold (60th percentile), R = avg_rating, C = global mean

C = df['avg_rating'].mean()
m = df['num_ratings'].quantile(0.60)

df['popularity_score'] = (
    (df['num_ratings'] / (df['num_ratings'] + m)) * df['avg_rating'] +
    (m / (df['num_ratings'] + m)) * C
).round(4)

print(f"Global mean rating  : {C:.3f}")
print(f"Min-vote threshold  : {m:,.0f}")
print()
print(df[['title','avg_rating','num_ratings','popularity_score']].head(5).to_string(index=False))

Global mean rating  : 3.916
Min-vote threshold  : 1,109

                                   title  avg_rating  num_ratings  popularity_score
Harry Potter and the Philosopher's Stone        4.48      7437005            4.4799
   Harry Potter and the Sorcerer's Stone        4.48      7434783            4.4799
                                Twilight        3.61      5173079            3.6101
                   To Kill a Mockingbird        4.28      4712812            4.2799
                   To Kill a Mockingbird        4.28      4712784            4.2799


In [24]:
# Count the number of awards a book has won (useful as a quality signal)
def count_awards(text):
    if not text: return 0
    return len([a for a in text.split(',') if a.strip()])

df['award_count'] = df['awards'].apply(count_awards)
print(f"Books with at least 1 award : {(df['award_count'] > 0).sum():,}")
print(f"Max awards for a single book: {df['award_count'].max()}")
print()
print(df[['title','award_count']].nlargest(5,'award_count').to_string(index=False))

Books with at least 1 award : 2,227
Max awards for a single book: 28

                                   title  award_count
Harry Potter and the Philosopher's Stone           28
   Harry Potter and the Sorcerer's Stone           28
                                Twilight           25
                               The Giver           22
                          The Book Thief           20


In [25]:
# Rename for clarity
df = df.rename(columns={
    'bookId'           : 'book_id',
    'author'           : 'authors',
    'num_pages'        : 'pages',
    'num_ratings'      : 'rating_count',
    'num_reviews'      : 'review_count',
    'avg_rating'       : 'rating',
})

# Logical column order
keep_order = [
    'book_id','title','authors','series','language',
    'description','genres','characters','places','awards',
    'rating','rating_count','review_count','popularity_score','award_count',
    'pages','first_publish_year','publish_year',
    'rated_1','rated_2','rated_3','rated_4','rated_5',
    'isbn','isbn13',
]
df = df[[c for c in keep_order if c in df.columns]]
print("Final columns:", df.columns.tolist())

Final columns: ['book_id', 'title', 'authors', 'series', 'language', 'description', 'genres', 'characters', 'places', 'awards', 'rating', 'rating_count', 'review_count', 'popularity_score', 'award_count', 'pages', 'first_publish_year', 'publish_year', 'rated_1', 'rated_2', 'rated_3', 'rated_4', 'rated_5', 'isbn', 'isbn13']


In [26]:
df = df.reset_index(drop=True)

assert df['title'].isna().sum()   == 0,  "Null titles remain!"
assert df['authors'].isna().sum() == 0,  "Null authors remain!"
assert df['rating'].between(0.5, 5).all(), "Rating out of range!"
assert (df['pages'] > 0).all(),            "Zero-page books remain!"
assert (df['rating_count'] > 0).all(),     "Zero-rating-count books remain!"
print("All assertions passed ✓")
print()
print(f"Final shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
df.sample(3)

All assertions passed ✓

Final shape : 16,122 rows × 25 columns



,book_id,title,authors,series,language,description,genres,characters,places,awards,...,pages,first_publish_year,publish_year,rated_1,rated_2,rated_3,rated_4,rated_5,isbn,isbn13
14585,12437,Dr. Zhivago: Curriculum Unit,Center for Learning,Standalone,English,,"Art,Biography,Business,Children's,Christian,Cl...",,,,...,90,1990.0,1990,0,0,0,0,0,1560771356,9781560771357
12201,14416,William Golding's Lord of the Flies,Terence Dewsnap,Monarch Notes,English,"Clear and to the point, Monarch Notes provide ...","Art,Biography,Business,Children's,Christian,Cl...",,,,...,256,NaN,1989,1,4,8,8,3,0671006169,9780671006167
14168,3809,Advanced Color Correction And Effects In Final...,Alexis Van Hurkman,Standalone,English,Final Cut Pro is a serious tool for serious us...,"Art,Biography,Business,Children's,Christian,Cl...",,,,...,634,2005.0,2006,0,0,1,3,2,0321335481,9780321335487


In [27]:
df.to_csv('../data/processed/Books_Cleaned.csv', index=False)